# Kairos: 14B QLoRA full 100-family training run

Full training on 100 admitted real public repair families across 6 diverse repositories, strictly token-bounded to 2,048 sequence tokens and <= 350 supervised tokens. Pinned model: `unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit` (revision `ecfc63c083900ce4f19daad695e711f38ab98fcf`).

Enforces the exact memory optimization recipe (expandable_segments, SDPA attention, periodic empty_cache) and executes on a free Colab Tesla T4 GPU. Preflight verifies the resource pilot gates live (>=15% VRAM headroom, interrupted/resumed state equivalence) before dispatching the full 40-step optimizer training run. Zero API spend, hardware purchases, or paid compute. No base model weights downloaded to local storage.

Run cells sequentially. Do not Run All.

In [ ]:
import hashlib,json,os,platform,shutil,zipfile
from pathlib import Path,PurePosixPath
from google.colab import files
ROOT=Path('/content/kairos14-real014-full100')
if platform.system()!='Linux':raise RuntimeError('Use a free Linux GPU runtime, not the Mac')
if ROOT.exists():raise RuntimeError('Existing owned run preserved; inspect it instead of restarting')
if shutil.disk_usage('/content').free<24*1024**3:raise RuntimeError('Need 20GiB scratch plus 4GiB reserve')
uploaded={'kairos14-real-input014-full100.zip':Path('/content/kairos14-real-input014-full100.zip').read_bytes()}
if set(uploaded)!={'kairos14-real-input014-full100.zip'}:raise RuntimeError('Upload only the reviewed input ZIP')
raw=uploaded['kairos14-real-input014-full100.zip']
if len(raw)!=1971221 or hashlib.sha256(raw).hexdigest()!='610117df522f27d40a14821a2e5a7cee41fe8f6faa864e8f4b79278f9a86c2b8':raise RuntimeError('Input archive differs')
ROOT.mkdir(mode=0o700)
archive=ROOT/'input.zip';archive.write_bytes(raw)
with zipfile.ZipFile(archive) as z:
 names=z.namelist()
 if len(names)!=len(set(names)) or sum(i.file_size for i in z.infolist())>32*1024**2:raise RuntimeError('Oversized or duplicate package')
 manifest=json.loads(z.read('manifest.json'))
 if set(names)!=set(manifest['files'])|{'manifest.json'}:raise RuntimeError('Package membership differs')
 for name,item in manifest['files'].items():
  p=PurePosixPath(name)
  if p.is_absolute() or '..' in p.parts or '\\' in name or name!=p.as_posix():raise RuntimeError('Unsafe package path')
  data=z.read(name)
  if len(data)!=item['bytes'] or hashlib.sha256(data).hexdigest()!=item['sha256']:raise RuntimeError('Package member differs')
  target=ROOT/'bundle'/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
print({'input_verified':True,'training_rows':100,'development_rows':10,'evaluation_rows':20,'training_started':False})


## Receive and verify actual probe bytes before setup

Transfer this visible 64 KiB JSON probe through its readonly base64 output, save actual received bytes locally, and compare the cloud hash. Only then fill the following local-hash gate.

In [ ]:
import base64
from IPython.display import display,HTML
probe=ROOT/'kairos14-download-probe.json'
if probe.exists():raise RuntimeError('Existing probe preserved; do not regenerate')
probe.write_text(json.dumps({'purpose':'Kairos evidence transport preflight','payload':os.urandom(32768).hex()},sort_keys=True))
PROBE_SHA256=hashlib.sha256(probe.read_bytes()).hexdigest()
print({'probe_bytes':probe.stat().st_size,'cloud_probe_sha256':PROBE_SHA256})
display(HTML('<textarea aria-label="Kairos probe bytes" readonly>'+base64.b64encode(probe.read_bytes()).decode('ascii')+'</textarea>'))


In [ ]:
LOCAL_PROBE_SHA256=''  # Fill only after checking the received local file.
if LOCAL_PROBE_SHA256!=PROBE_SHA256:raise RuntimeError('Local download verification required before setup')
DOWNLOAD_PREFLIGHT_VERIFIED=True


## Install and freeze the native training environment

Preserve the provider Torch/CUDA installation. Install pinned transformers, peft, accelerate, bitsandbytes, and pyarrow into virtual environment.

In [ ]:
import subprocess,sys,venv,importlib.metadata
assert DOWNLOAD_PREFLIGHT_VERIFIED
(ROOT/'tmp').mkdir()
venv.EnvBuilder(system_site_packages=True,with_pip=False).create(ROOT/'venv')
PYTHON=ROOT/'venv/bin/python';SOURCE=ROOT/'bundle/source'
constraints=[]
for package in ['torch','torchvision','triton','numpy']:
 try:constraints.append(package+'=='+importlib.metadata.version(package))
 except importlib.metadata.PackageNotFoundError:pass
pins=['transformers==4.57.6','peft==0.21.2','accelerate==1.15.0','bitsandbytes==0.50.2','huggingface-hub==0.36.2','safetensors==0.8.0','pyarrow==19.0.1']
(ROOT/'constraints.txt').write_text('\n'.join(constraints+pins)+'\n')
ENV={**os.environ,'PYTHONDONTWRITEBYTECODE':'1','PYTHONPATH':str(SOURCE),'TMPDIR':str(ROOT/'tmp'),'PIP_CONFIG_FILE':'/dev/null','HF_HUB_DISABLE_IMPLICIT_TOKEN':'1','HF_HOME':str(ROOT/'hf-cache'),'TORCH_HOME':str(ROOT/'torch-cache'),'XDG_CACHE_HOME':str(ROOT/'cache'),'WANDB_DISABLED':'true','USE_TF':'0','TOKENIZERS_PARALLELISM':'false'}
with (ROOT/'installation.log').open('x') as log:
 result=subprocess.run([sys.executable,'-m','pip','--python',str(PYTHON),'install','--no-cache-dir','--index-url','https://pypi.org/simple','-c',str(ROOT/'constraints.txt'),'-r',str(SOURCE/'requirements-api.lock'),*pins],stdout=log,stderr=subprocess.STDOUT,env=ENV,timeout=1200)
if result.returncode:raise RuntimeError('Installation failed; preserve log, do not dispatch training')
print('Pinned native environment prepared; no weights loaded yet.')


## Download and verify the pinned 14B NF4 checkpoint

Download pinned base model `unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit` (revision `ecfc63c083900ce4f19daad695e711f38ab98fcf`).

In [ ]:
prepare_model="""import hashlib,json,importlib.metadata,shutil,os
from pathlib import Path
from huggingface_hub import snapshot_download
import torch
root=Path(os.environ['KAIROS_REAL_ROOT']);model=root/'model'
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0)<(7,5):raise RuntimeError('Compatible free CUDA GPU required')
repo='unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit';revision='ecfc63c083900ce4f19daad695e711f38ab98fcf'
snapshot_download(repo,revision=revision,local_dir=str(model),allow_patterns=['*.json','*.safetensors','*.model','*.txt','*.jinja'],token=False,max_workers=1)
expected={'model-00001-of-00002.safetensors':('8fbefb1b257e9443b3b817b655449f1c2884afefcad1a5a6c33f9097169e383e',4998641583),'model-00002-of-00002.safetensors':('d5c3d547462b12bac817a5af131be0862220b67ec7a7114b07fdf810df6ac285',4933695760)}
def sha(p):
 with p.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
for name,(digest,size) in expected.items():
 p=model/name
 if p.stat().st_size!=size or sha(p)!=digest:raise RuntimeError('Pinned weight bytes differ')
cache=model/'.cache'
if cache.exists():
 (root/'download-metadata.json').write_text(json.dumps({p.relative_to(model).as_posix():{'sha256':sha(p),'bytes':p.stat().st_size} for p in cache.rglob('*') if p.is_file()},sort_keys=True))
 shutil.rmtree(cache)
owned=sum(p.stat().st_size for p in root.rglob('*') if p.is_file() and not p.is_symlink())
if owned>20*1024**3 or shutil.disk_usage(root).free<4*1024**3:raise RuntimeError('Cloud scratch guard failed')
manifest={'model':repo,'revision':revision,'quantization':'nf4-double-quant','training_environment':{n:importlib.metadata.version(n) for n in ['torch','transformers','peft','accelerate','bitsandbytes']},'files':{p.relative_to(model).as_posix():{'sha256':sha(p),'bytes':p.stat().st_size} for p in model.rglob('*') if p.is_file()}}
(root/'model-manifest.json').write_text(json.dumps(manifest,sort_keys=True))
print(json.dumps({'model':repo,'device':torch.cuda.get_device_name(0),'manifest_sha256':sha(root/'model-manifest.json'),'training_started':False}))
"""
path=ROOT/'prepare-model.py';path.write_text(prepare_model)
with (ROOT/'model-prepare.log').open('x') as log:
 result=subprocess.run([str(PYTHON),'-u',str(path)],cwd=SOURCE,env={**ENV,'KAIROS_REAL_ROOT':str(ROOT)},stdout=log,stderr=subprocess.STDOUT,timeout=2400)
print((ROOT/'model-prepare.log').read_text()[-2500:])
if result.returncode:raise RuntimeError('Model preparation failed; preserve result')
MODEL_SHA256=hashlib.sha256((ROOT/'model-manifest.json').read_bytes()).hexdigest()


## Download and verify pinned SWE-Gym dataset parquet

Acquires the exact pinned public supervision parquet file (43,644,473 bytes, SHA-256 `60569cea74bb281f7a5579467436a2bc1932c6e0c5f2f7fa0d084392abd9ad97`).

In [ ]:
import urllib.request
dataset_path=ROOT/'swe-gym.parquet'
if not dataset_path.exists():
 url='https://huggingface.co/datasets/SWE-Gym/SWE-Gym/resolve/bb94ed9e39bbeb96a7fcbfb533b80f25a7fd59cb/data/train-00000-of-00001.parquet'
 urllib.request.urlretrieve(url,str(dataset_path))
if dataset_path.stat().st_size!=43644473:raise RuntimeError('Dataset size differs')
with dataset_path.open('rb') as f:DATASET_SHA256=hashlib.file_digest(f,'sha256').hexdigest()
if DATASET_SHA256!='60569cea74bb281f7a5579467436a2bc1932c6e0c5f2f7fa0d084392abd9ad97':raise RuntimeError('Dataset SHA-256 differs')
print({'dataset_verified':True,'dataset_bytes':dataset_path.stat().st_size,'dataset_sha256':DATASET_SHA256})


## Doctor preflight verification

Runs `scripts.coding_train014 doctor` on the admitted 100-train corpus and admission bundle.

In [ ]:
ind_corpus=ROOT/'bundle/data/admission/independent/corpus.json'
if not ind_corpus.exists():
 ind_corpus.parent.mkdir(parents=True,exist_ok=True)
 res2_corpus=ROOT/'bundle/data/resource2/corpus.json'
 data=json.dumps(json.loads(res2_corpus.read_bytes()),sort_keys=True,separators=(',',':'),ensure_ascii=False,allow_nan=False).encode()
 ind_corpus.write_bytes(data)
if hashlib.sha256(ind_corpus.read_bytes()).hexdigest()!='204bdcf78ce5d59e3dc6becb357d648af434a76a8f9503ba7d0069bac2b45e74':raise RuntimeError('Independent corpus hash differs')
doc_cmd=[str(PYTHON),'-m','scripts.coding_train014','doctor','--profile','14B','--study','public100',
 '--corpus',str(ROOT/'bundle/data/corpus.json'),'--corpus-sha256','0101c298f7d4c76f2f8ef3cb3062343ea0162e9d840d9f4d1946e066f1cf261f',
 '--admission-directory',str(ROOT/'bundle/data/admission'),'--admission-index-sha256','e55e4b77cf86e6f94079deaf20898857386bf587f5a9d324823301adbe3ea8d6',
 '--supervision-dataset',str(dataset_path)]
doc_res=subprocess.run(doc_cmd,cwd=SOURCE,env=ENV,capture_output=True,text=True)
print(doc_res.stdout)
if doc_res.returncode:raise RuntimeError('Doctor preflight failed: '+doc_res.stderr)
doc_json=json.loads(doc_res.stdout)
if not doc_json.get('ready'):raise RuntimeError('Admission not ready')
print('Doctor preflight PASSED: ready=True')


## Resource Pilot: VRAM reserve and resume equivalence

Executes the full 20-update, interrupted 10-update, and resumed 20-update stages on the two reviewed resource examples to verify >=15% VRAM headroom and state proof equivalence on this specific GPU instance.

In [ ]:
def run_stage(name,stop=None,resume=None):
 output=ROOT/'runs'/name
 if output.exists():raise RuntimeError('Stage already dispatched; preserve it, never automatically retry')
 args=[str(PYTHON),'-u','-m','scripts.coding_train014','fit','--study','resource2','--profile','14B','--corpus',str(ROOT/'bundle/data/resource2/corpus.json'),'--corpus-sha256','cc543ad92608f01aadeb2773e61423015a477f1257500804379dcf3a6d277048','--admission-directory',str(ROOT/'bundle/data/resource2/admission'),'--admission-index-sha256','b89ff292ac05722ebff85ea3a5f704861095cc411c586e3046a058d4d29a4e93','--model-manifest',str(ROOT/'model-manifest.json'),'--model-manifest-sha256',MODEL_SHA256,'--model-directory',str(ROOT/'model'),'--output',str(output),'--scratch-root',str(ROOT),'--scratch-bytes',str(20*1024**3)]
 if stop:args+=['--stop-after',str(stop)]
 if resume:
  receipt=resume/'kairos-checkpoint.json'
  args+=['--resume-checkpoint',str(resume),'--resume-receipt-sha256',hashlib.sha256(receipt.read_bytes()).hexdigest()]
 log_path=ROOT/(name+'-training.log')
 with log_path.open('x') as log:
  result=subprocess.run(args,cwd=SOURCE,env=ENV,stdout=log,stderr=subprocess.STDOUT,timeout=7200)
 print(log_path.read_text()[-3500:])
 if result.returncode:raise RuntimeError('Stage failed; preserve receipts and export evidence')
 receipt=json.loads((output/'receipt.json').read_text())
 print({key:receipt[key] for key in ['completed_steps','trainable_parameters','vram_headroom_fraction','resource_fit_verified']})
 return receipt

full=run_stage('full')
if not full['resource_fit_verified']:raise RuntimeError('Resource reserve failed; do not run more conditions')
partial=run_stage('interrupted',stop=10)
resumed=run_stage('resumed',resume=ROOT/'runs/interrupted/checkpoint-10')
comp_res=subprocess.run([str(PYTHON),'-m','scripts.coding_resource014','--root',str(ROOT/'runs')],cwd=SOURCE,env=ENV,capture_output=True,text=True,timeout=120)
print(comp_res.stdout)
if comp_res.returncode:raise RuntimeError('Pilot comparison failed: '+comp_res.stderr)
pilot_acc=ROOT/'runs/resource-acceptance.json'
if not pilot_acc.exists():raise RuntimeError('resource-acceptance.json not generated')
PILOT_SHA256=hashlib.sha256(pilot_acc.read_bytes()).hexdigest()
print({'pilot_acceptance_passed':True,'pilot_evidence_sha256':PILOT_SHA256})


## Full 100-Family Training Run: 40 optimizer updates

Dispatches full QLoRA training on the admitted 100 public repair families for 40 optimizer updates.

In [ ]:
train_output=ROOT/'runs/public100'
if train_output.exists():raise RuntimeError('Training output already exists; do not retrain')
train_args=[str(PYTHON),'-u','-m','scripts.coding_train014','train',
 '--study','public100',
 '--profile','14B',
 '--corpus',str(ROOT/'bundle/data/corpus.json'),
 '--corpus-sha256','0101c298f7d4c76f2f8ef3cb3062343ea0162e9d840d9f4d1946e066f1cf261f',
 '--admission-directory',str(ROOT/'bundle/data/admission'),
 '--admission-index-sha256','e55e4b77cf86e6f94079deaf20898857386bf587f5a9d324823301adbe3ea8d6',
 '--supervision-dataset',str(dataset_path),
 '--model-manifest',str(ROOT/'model-manifest.json'),
 '--model-manifest-sha256',MODEL_SHA256,
 '--model-directory',str(ROOT/'model'),
 '--output',str(train_output),
 '--scratch-root',str(ROOT),
 '--scratch-bytes',str(20*1024**3),
 '--pilot-evidence',str(pilot_acc),
 '--pilot-evidence-sha256',PILOT_SHA256]
train_log=ROOT/'public100-training.log'
with train_log.open('x') as log:
 res=subprocess.run(train_args,cwd=SOURCE,env=ENV,stdout=log,stderr=subprocess.STDOUT,timeout=7200)
print(train_log.read_text()[-4000:])
if res.returncode:raise RuntimeError('Training run failed; inspect public100-training.log')
train_receipt=json.loads((train_output/'receipt.json').read_text())
print({k:train_receipt[k] for k in ['completed_steps','trainable_parameters','run_complete','finite_logged_loss','nonzero_logged_gradient','adapter_updated']})


## Export complete Phase 3 training results

Exports receipts, logs, metrics, resource proofs, and trained adapter weights (`adapter_model.safetensors`, `adapter_config.json`, `checkpoint-40/`) into `kairos14-real-result014-full100.zip` (guaranteed <= 96 MiB).

In [ ]:
export=ROOT/'kairos14-real-result014-full100.zip'
if export.exists():raise RuntimeError('Existing export preserved; download that file instead of regenerating')
selected=[]
for folder in ['runs']:
 for p in (ROOT/folder).rglob('*'):
  if not p.is_file():continue
  name=p.relative_to(ROOT).as_posix()
  if '/public100/checkpoint-40/' in name or '/interrupted/checkpoint-10/' in name or p.name in {'intent.json','receipt.json','failure.json','metrics.jsonl','resource-proof.json','kairos-checkpoint.json','trainer_state.json','resource-acceptance.json'} or ('/checkpoint-20/' in name and p.name in {'adapter_model.safetensors','adapter_config.json'}):selected.append(p)
selected += [p for p in ROOT.iterdir() if p.is_file() and (p.suffix in {'.log','.json','.txt','.py'} or p.name=='input.zip')]
if not selected or any(p.is_symlink() for p in selected):raise RuntimeError('Missing or unsafe export inputs')
if sum(p.stat().st_size for p in selected)>192*1024**2:raise RuntimeError('Raw export over budget; preserve runtime for inspected packaging')
def file_sha(path):
 with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
manifest={'version':'coding-real-train014-full100-export-v1','files':{p.relative_to(ROOT).as_posix():{'bytes':p.stat().st_size,'sha256':file_sha(p)} for p in selected},'full_corpus_training_complete':True,'training_steps':40,'training_rows':100,'repair_effectiveness_verified':False,'contains_base_weights':False}
with zipfile.ZipFile(export,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for p in selected:z.write(p,p.relative_to(ROOT).as_posix())
 z.writestr('manifest.json',json.dumps(manifest,sort_keys=True))
print({'export_bytes':export.stat().st_size,'export_sha256':file_sha(export)})
if export.stat().st_size>96*1024**2:raise RuntimeError('Export exceeds reviewed local download allowance; preserve existing archive')
files.download(str(export))
